In [ ]:
import os
import nibabel as nib
import numpy as np

# === Dossier des masques ===
mask_dir = "/home/amenacer/Stage/data-4D/7/$FRAME"

# === Création du dossier de sortie ===
output_dir = os.path.join(mask_dir, "merged_masks")
os.makedirs(output_dir, exist_ok=True)

# === Récupération des masques du ventricule gauche (VG) ===
mask1_files = sorted([f for f in os.listdir(mask_dir) if f.endswith("_ROIMask-1.nii")])

for mask1_file in mask1_files:
    base_name = mask1_file.replace("_ROIMask-1.nii", "")
    mask2_file = f"{base_name}_ROIMask-2.nii"

    path1 = os.path.join(mask_dir, mask1_file)
    path2 = os.path.join(mask_dir, mask2_file)

    # Vérification que les deux fichiers existent
    if not os.path.exists(path2):
        print(f"⛔ Masque 2 manquant pour {base_name}, ignoré.")
        continue

    # Chargement des deux masques
    img1 = nib.load(path1)
    img2 = nib.load(path2)
    data1 = img1.get_fdata()
    data2 = img2.get_fdata()

    # Fusion : VG = 1, Myocarde = 2
    merged = np.zeros_like(data1)
    merged[data1 > 0] = 1
    merged[data2 > 0] = 2

    # Création de l'image fusionnée
    merged_img = nib.Nifti1Image(merged, img1.affine, img1.header)

    # Sauvegarde
    output_path = os.path.join(output_dir, f"{base_name}_merged.nii.gz")
    nib.save(merged_img, output_path)
    print(f"✅ Fusion sauvegardée : {base_name}_merged.nii.gz")

print("\n🎉 Fusion terminée !")
